In [1]:
import requests
import os
import sys
import platform
from lakehouse import bronze
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import DeltaTable, configure_spark_with_delta_pip
from delta.tables import DeltaMergeBuilder
from pyspark.sql import DataFrame

In [2]:
if platform.system() == "Windows":
    os.environ["PYSPARK_PYTHON"] = sys.executable
    os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
    print("Adding Python ENV variables on Windows")

Adding Python ENV variables on Windows


In [3]:
builder = (
    SparkSession.builder.appName("Data with Nikk the Greek Spark Session")
    .master("local[4]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [4]:
CATALOG = spark.catalog.currentCatalog()

# 1. Set Up

In [5]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")

DataFrame[]

In [6]:
options = {
    "catalog": CATALOG,
    "target_schema": "bronze",
}

# 2 Overwrite

In [7]:
class StarWarsBronze(bronze.Bronze):
    def custom_load(self, table):
        results = []
        query = f"https://swapi.tech/api/{table}"
        json_request = requests.get(query).json()
        results.extend(json_request["results"])

        while json_request["next"]:
            json_request = requests.get(json_request["next"]).json()
            results.extend(json_request["results"])
        return spark.createDataFrame(results)


bronze_instance = StarWarsBronze(spark, **options)

In [9]:
(
    bronze_instance.load()
    .transform()
    .write(mode="overwrite")
    .tblproperties(clusterby=["LH_BronzeTS"])
    .optimize(optimize=True, vacuum=True, analyze=False, excl_cols=["url"])
    .execute("people", "planets")
)
spark.sql(f"SELECT * FROM {CATALOG}.bronze.people").show(truncate=False)

+--------------------------+--------------+---+------------------------------------+
|LH_BronzeTS               |name          |uid|url                                 |
+--------------------------+--------------+---+------------------------------------+
|2025-02-17 18:48:16.319493|Quarsh Panaka |42 |https://www.swapi.tech/api/people/42|
|2025-02-17 18:48:16.319493|Shmi Skywalker|43 |https://www.swapi.tech/api/people/43|
|2025-02-17 18:48:16.319493|Darth Maul    |44 |https://www.swapi.tech/api/people/44|
|2025-02-17 18:48:16.319493|Bib Fortuna   |45 |https://www.swapi.tech/api/people/45|
|2025-02-17 18:48:16.319493|Ayla Secura   |46 |https://www.swapi.tech/api/people/46|
|2025-02-17 18:48:16.319493|Ratts Tyerel  |47 |https://www.swapi.tech/api/people/47|
|2025-02-17 18:48:16.319493|Dud Bolt      |48 |https://www.swapi.tech/api/people/48|
|2025-02-17 18:48:16.319493|Gasgano       |49 |https://www.swapi.tech/api/people/49|
|2025-02-17 18:48:16.319493|Ben Quadinaros|50 |https://www.swapi.

In [16]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.bronze.people")
print(f"No. Rows: {sdf.count()}")
sdf.show(100)

No. Rows: 82
+--------------------+--------------------+---+--------------------+
|         LH_BronzeTS|                name|uid|                 url|
+--------------------+--------------------+---+--------------------+
|2025-02-17 18:48:...|       Quarsh Panaka| 42|https://www.swapi...|
|2025-02-17 18:48:...|      Shmi Skywalker| 43|https://www.swapi...|
|2025-02-17 18:48:...|          Darth Maul| 44|https://www.swapi...|
|2025-02-17 18:48:...|         Bib Fortuna| 45|https://www.swapi...|
|2025-02-17 18:48:...|         Ayla Secura| 46|https://www.swapi...|
|2025-02-17 18:48:...|        Ratts Tyerel| 47|https://www.swapi...|
|2025-02-17 18:48:...|            Dud Bolt| 48|https://www.swapi...|
|2025-02-17 18:48:...|             Gasgano| 49|https://www.swapi...|
|2025-02-17 18:48:...|      Ben Quadinaros| 50|https://www.swapi...|
|2025-02-17 18:48:...|          Mace Windu| 51|https://www.swapi...|
|2025-02-17 18:48:...|        Ki-Adi-Mundi| 52|https://www.swapi...|
|2025-02-17 18:48:...

In [17]:
sdf = spark.sql(f"SELECT * FROM {CATALOG}.bronze.planets")
print(f"No. Rows: {sdf.count()}")
sdf.show(100)

No. Rows: 60
+--------------------+--------------+---+--------------------+
|         LH_BronzeTS|          name|uid|                 url|
+--------------------+--------------+---+--------------------+
|2025-02-17 18:49:...|       Mygeeto| 16|https://www.swapi...|
|2025-02-17 18:49:...|       Felucia| 17|https://www.swapi...|
|2025-02-17 18:49:...|Cato Neimoidia| 18|https://www.swapi...|
|2025-02-17 18:49:...|     Saleucami| 19|https://www.swapi...|
|2025-02-17 18:49:...|       Stewjon| 20|https://www.swapi...|
|2025-02-17 18:49:...|        Eriadu| 21|https://www.swapi...|
|2025-02-17 18:49:...|      Corellia| 22|https://www.swapi...|
|2025-02-17 18:49:...|         Rodia| 23|https://www.swapi...|
|2025-02-17 18:49:...|     Nal Hutta| 24|https://www.swapi...|
|2025-02-17 18:49:...|     Dantooine| 25|https://www.swapi...|
|2025-02-17 18:49:...|    Bestine IV| 26|https://www.swapi...|
|2025-02-17 18:49:...|   Ord Mantell| 27|https://www.swapi...|
|2025-02-17 18:49:...|       unknown| 28|h

In [18]:
bronze_instance.data["people"].show()

+--------------------+--------------------+---+--------------------+
|         LH_BronzeTS|                name|uid|                 url|
+--------------------+--------------------+---+--------------------+
|2025-02-17 18:52:...|      Luke Skywalker|  1|https://www.swapi...|
|2025-02-17 18:52:...|               C-3PO|  2|https://www.swapi...|
|2025-02-17 18:52:...|               R2-D2|  3|https://www.swapi...|
|2025-02-17 18:52:...|         Darth Vader|  4|https://www.swapi...|
|2025-02-17 18:52:...|         Leia Organa|  5|https://www.swapi...|
|2025-02-17 18:52:...|           Owen Lars|  6|https://www.swapi...|
|2025-02-17 18:52:...|  Beru Whitesun lars|  7|https://www.swapi...|
|2025-02-17 18:52:...|               R5-D4|  8|https://www.swapi...|
|2025-02-17 18:52:...|   Biggs Darklighter|  9|https://www.swapi...|
|2025-02-17 18:52:...|      Obi-Wan Kenobi| 10|https://www.swapi...|
|2025-02-17 18:52:...|    Anakin Skywalker| 11|https://www.swapi...|
|2025-02-17 18:52:...|      Wilhuf

In [19]:
bronze_instance.data["planets"].show()

+--------------------+--------------+---+--------------------+
|         LH_BronzeTS|          name|uid|                 url|
+--------------------+--------------+---+--------------------+
|2025-02-17 18:52:...|      Tatooine|  1|https://www.swapi...|
|2025-02-17 18:52:...|      Alderaan|  2|https://www.swapi...|
|2025-02-17 18:52:...|      Yavin IV|  3|https://www.swapi...|
|2025-02-17 18:52:...|          Hoth|  4|https://www.swapi...|
|2025-02-17 18:52:...|       Dagobah|  5|https://www.swapi...|
|2025-02-17 18:52:...|        Bespin|  6|https://www.swapi...|
|2025-02-17 18:52:...|         Endor|  7|https://www.swapi...|
|2025-02-17 18:52:...|         Naboo|  8|https://www.swapi...|
|2025-02-17 18:52:...|     Coruscant|  9|https://www.swapi...|
|2025-02-17 18:52:...|        Kamino| 10|https://www.swapi...|
|2025-02-17 18:52:...|      Geonosis| 11|https://www.swapi...|
|2025-02-17 18:52:...|        Utapau| 12|https://www.swapi...|
|2025-02-17 18:52:...|      Mustafar| 13|https://www.sw

# 6 Clean Up

In [27]:
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.bronze CASCADE")

DataFrame[]